In [ ]:
from torchvision import transforms
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image as PILImage
import numpy as np
import os
import pandas as pd
import random
from tqdm.auto import tqdm

In [ ]:
import quantus
from quantus import PointingGame, MaxSensitivity
import glob
from PIL import Image

from pytorch_grad_cam.utils.image import preprocess_image, show_cam_on_image
from quantus import norm_func, perturb_func, similarity_func
from datetime import datetime

from xai_lib import MemoryManagement as mm
from xai_lib import AttributionEngine as aengine

In [ ]:
name = "aveera"
PREPROCESSED_DIR = f"/data/{name}/preprocessed_idrid"

In [ ]:
_engine_cache = {}

def get_engine_for(model):
    """Look up the AttributionEngine that was built for this exact model object."""

    eng = _engine_cache.get(id(model))
    if eng is None:
        raise RuntimeError(
            "No AttributionEngine registered for this model."
        )
    return eng

In [ ]:
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Lets cuDNN pick the fastest conv algorithms since our input size (224x224) is fixed
torch.backends.cudnn.benchmark = True

In [ ]:
def get_predicted_labels(model, samples, device, transform=test_transform):
    """Recomputes the model's predicted class per sample."""
    preds = []
    model.eval()
    with torch.no_grad():
        for raw_img_np, _ in samples:
            input_tensor = transform(PILImage.fromarray(raw_img_np)).unsqueeze(0).to(device)
            pred = model(input_tensor).argmax(dim=1).item()
            preds.append(pred)
    return np.array(preds)

In [ ]:
def explain_gradcam(model, inputs, targets, **kwargs):
    maps = []
    eng = get_engine_for(model)
    for img, target in zip(inputs, targets):
        img_tensor = torch.from_numpy(img).float()
        gradcam = eng.compute_gradcam(img_tensor.unsqueeze(0).to(eng.device), int(target))
        maps.append(gradcam)
        del img_tensor
    return np.stack(maps)[:, None, :, :]


def explain_lime(model, inputs, targets, **kwargs):
    maps = []
    eng = get_engine_for(model)
    for img, target in zip(inputs, targets):
        img_tensor = torch.from_numpy(img).float()
        raw = eng.unnormalize_to_uint8(img_tensor)
        explanation = eng.compute_lime(raw, int(target))
        heatmap = eng.get_lime_heatmap(explanation, int(target))
        maps.append(heatmap)
        eng.close_lime_explanation(explanation)
        del explanation, img_tensor, raw
    return np.stack(maps)[:, None, :, :]


def explain_shap(model, inputs, targets, **kwargs):
    maps = []
    eng = get_engine_for(model)
    for img, target in zip(inputs, targets):
        img_tensor = torch.from_numpy(img).float()
        raw = eng.unnormalize_to_uint8(img_tensor)
        shap_values = eng.compute_shap(raw, target_class=int(target), all_classes=False)
        heatmap, _ = eng.get_shap_heatmap(shap_values, class_position=0)
        maps.append(heatmap)
        del shap_values, img_tensor, raw
    return np.stack(maps)[:, None, :, :]

explain_funcs = {
    "GradCAM": explain_gradcam,
    "LIME": explain_lime,
    "SHAP": explain_shap,
}

In [ ]:
drive_model_dir = "/data/aveera/models"
RESULTS_DIR = "/data/aveera/results"
CHECKPOINT_DIR = "/data/aveera/results"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "maxsens_per_image_checkpoint.csv")

CHECKPOINT_COLUMNS = ["model", "method", "image_index", "max_sensitivity"]


def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=CHECKPOINT_COLUMNS)


def append_checkpoint_row(row: dict):
    """Append a single completed (model, method, image_index) result to disk immediately."""
    df_row = pd.DataFrame([row], columns=CHECKPOINT_COLUMNS)
    write_header = not os.path.exists(CHECKPOINT_PATH)
    df_row.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)


def already_done(checkpoint_df, model_name, method, image_index):
    if checkpoint_df.empty:
        return False
    match = checkpoint_df[
        (checkpoint_df["model"] == model_name)
        & (checkpoint_df["method"] == method)
        & (checkpoint_df["image_index"] == image_index)
    ]
    return not match.empty


**Max-sensitivity configuration**

In [ ]:
MAXSENS_KWARGS = dict(
    nr_samples=10,
    lower_bound=0.2,
    norm_numerator=norm_func.fro_norm,
    norm_denominator=norm_func.fro_norm,
    similarity_func=similarity_func.difference,
)

MODEL_NAMES = ["resnet50", "efficientnetb0"]
METHOD_NAMES = ["GradCAM", "LIME", "SHAP"]  # processed strictly one at a time, in this order


**Running max-sensitivity**

In [ ]:
def load_preprocessed_from_disk(save_dir):
    manifest_path = os.path.join(save_dir, "manifest.csv")
    if not os.path.exists(manifest_path):
        raise FileNotFoundError(
            f"No manifest.csv found in {save_dir}"
        )
    manifest = pd.read_csv(manifest_path).sort_values("index")

    samples = []
    for _, row in manifest.iterrows():
        img = np.array(Image.open(row["image_path"]).convert('RGB'))
        mask = np.array(Image.open(row["mask_path"]).convert('L'))
        samples.append((img, mask))
    return samples

In [ ]:
N_TEST = 81
SEED=42
maxsens_samples = load_preprocessed_from_disk(PREPROCESSED_DIR)
maxsens_samples = maxsens_samples[:N_TEST]

checkpoint_df = load_checkpoint()
print(f"Resuming from checkpoint with {len(checkpoint_df)} previously completed (model, method, image) rows.")

# Fast O(1) membership check instead of re-scanning/rebuilding checkpoint_df every image.
completed_keys = set(zip(checkpoint_df["model"], checkpoint_df["method"], checkpoint_df["image_index"]))

for model_name in MODEL_NAMES:
    print(f"\n{'='*60}\n  MaxSensitivity | Model: {model_name}\n{'='*60}")

    # ---- Load ONE model at a time ----
    drive_weight_path = os.path.join(drive_model_dir, f"{model_name}_best_model.pth")
    model = mm.load_model(model_name, drive_weight_path, device)

    engine = aengine.AttributionEngine(model, device)
    _engine_cache[id(model)] = engine

    # Build the normalized input batch 
    input_tensors = []
    for raw_img_np, _ in maxsens_samples:
        image_np = raw_img_np.astype(np.float32) / 255.0
        t = preprocess_image(image_np, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        input_tensors.append(t.squeeze(0))
    input_batch = torch.stack(input_tensors)
    x_batch_all = input_batch.cpu().numpy()
    del input_tensors, input_batch
    mm.clear_memory()

    y_batch_all = get_predicted_labels(model, maxsens_samples, device)
    mm.gpu_mem_report(f"after loading {model_name}")

    # ---- Process ONE XAI method at a time ----
    for method in METHOD_NAMES:
        explain_func = explain_funcs[method]
        print(f"\n  Method: {method} ({model_name})")

        metric = MaxSensitivity(**MAXSENS_KWARGS)

        n_images = x_batch_all.shape[0]
        pbar = tqdm(range(n_images), desc=f"MaxSensitivity [{model_name}/{method}]")

        for img_idx in pbar:
            if (model_name, method, img_idx) in completed_keys:
                continue  # already computed in a previous (crashed) run

            np.random.seed(SEED)
            random.seed(SEED)
            torch.manual_seed(SEED)

            x_single = x_batch_all[img_idx:img_idx + 1]
            y_single = y_batch_all[img_idx:img_idx + 1]
            a_single = None

            try:
                a_single = explain_func(model, x_single, y_single)

                score = metric(
                    model=model,
                    x_batch=x_single,
                    y_batch=y_single,
                    a_batch=a_single,
                    device=device,
                    explain_func=explain_func,
                )[0]

                row = {
                    "model": model_name,
                    "method": method,
                    "image_index": img_idx,
                    "max_sensitivity": float(score),
                }
                append_checkpoint_row(row)
                completed_keys.add((model_name, method, img_idx))

            except Exception as e:
                print(f"    Error on {model_name}/{method}/image {img_idx}: {e}")

            finally:
                # Clear Python/PyTorch memory after each image 
                a_single = None
                x_single = None
                y_single = None
                mm.clear_memory()
                if img_idx % 5 == 0:
                    mm.log_rss(f"{model_name}/{method}/image {img_idx}")

        # Clear the MaxSensitivity object after each  method
        del metric
        mm.clear_memory()

        # Reload once per method (cheap: ~hundreds of rows) instead of accumulating
        # checkpoint_df in memory via pd.concat on every single image.
        checkpoint_df = load_checkpoint()
        method_df = checkpoint_df[
            (checkpoint_df["model"] == model_name) & (checkpoint_df["method"] == method)
        ]
        method_summary_path = os.path.join(
            RESULTS_DIR, f"maxsens_{model_name}_{method}_summary.csv"
        )
        method_df.to_csv(method_summary_path, index=False)
        print(f"  Saved {len(method_df)} results for {model_name}/{method} -> {method_summary_path}")
        mm.gpu_mem_report(f"after {model_name}/{method}")

    del x_batch_all, y_batch_all

    # Clear the model completely before loading the next one 
    mm.unload_model(model=model, engine=engine, engine_cache=_engine_cache)
    mm.gpu_mem_report(f"after unloading {model_name}")

print("\nAll models and methods complete.")
